# The Seven Layers of AI: Layer 4: Deep Learning

This notebook contains every example program from **Chapter 5** of the study guide *The Seven Layers of AI*, in the order they appear. Each program comes with the explanation that introduces it in the guide, the output printed in the guide, what to notice in that output, and the Investigate tasks.

**How to use it:** run the cells from top to bottom. Each program is self-contained, so you can also run any single program on its own after the setup cell. Change the code freely: the Investigate tasks suggest where to start.

## Programs in this notebook

1. `l4_deep_signal.py`: signal propagation through 50 layers
2. `l4_convolution.py`: convolution from scratch
3. `l4_cnn_numpy.py`: a CNN trained from scratch in NumPy
4. `l4_rnn.py`: a character-level RNN
5. `l4_lstm.py`: how gates preserve memory
6. `l4_attention.py`: attention from scratch
7. `l4_autoencoder.py`: compression, denoising and anomaly detection
8. `l4_transfer_augment.py`: transfer learning and augmentation
9. `l4_keras_cnn.py`: CNNs and pretrained models in Keras
10. `l4_adversarial.py`: fooling a network

## Setup

The programs in this notebook use: `numpy scikit-learn matplotlib`. If any are missing, uncomment the line below and run it once.

In [1]:
# %pip install numpy scikit-learn matplotlib
%matplotlib inline

## 1. Signal propagation through 50 layers

**File:** `l4_deep_signal.py`

In [2]:
"""
Layer 4 example: why very deep networks were hard to train, and the three
techniques that fixed it.
 
File: l4_deep_signal.py
 
Purpose:
    A deep network multiplies its signal by a weight matrix at every layer.
    Over 50 layers even a small per-layer shrink or growth compounds, so the
    activations (forward) and gradients (backward) either VANISH towards zero
    or EXPLODE towards infinity, and the network cannot learn. Good
    INITIALISATION, NORMALISATION and RESIDUAL (skip) CONNECTIONS keep the
    signal at a healthy size, which is what made modern deep learning possible.
 
How it works:
    1. Push a batch of 512 random inputs forward through a 50-layer network.
    2. Measure the standard deviation of the activations at layer 50.
    3. Send a random gradient backwards with the chain rule and measure its
       average size when it reaches layer 1.
    4. Repeat for seven configurations and compare.
 
What to look for in the output:
    - Tiny weights (std 0.01): activations shrink to 5.936e-48 and the
      gradient to 4.551e-48, a vanishing signal.
    - Large weights with ReLU explode to 2.327e+52; with tanh the activations
      saturate (0.974) but the gradient still explodes to 8.039e+22.
    - He init with ReLU keeps both near 1 (2.269 and 0.953), and adding
      residual connections stays healthy (2.614 and 1.190).
    - Batch normalisation rescues even the tiny weights (std 0.622).
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
# Fixed seed so every run prints the same numbers.
rng = np.random.default_rng(0)
# Each layer has 256 units; the network is 50 layers deep.
WIDTH, DEPTH = 256, 50
x0 = rng.normal(0, 1, (512, WIDTH))                     # batch of 512 random inputs
 
 
# ---- run(): one forward and one backward pass through a 50-layer network --------
# init chooses how the weights are drawn, act chooses ReLU or tanh, and the two
# flags switch on batch normalisation or residual connections. Returns
# (activation std at the last layer, mean gradient size at the first layer).
#
# Why the initialisation matters: each unit sums WIDTH inputs times weights.
# If the weights have std s, the output variance is roughly WIDTH * s^2 times
# the input variance. Repeat that 50 times and any factor away from 1 compounds:
#   std 0.01 -> factor 256 * 0.0001 = 0.0256 per layer -> vanishes
#   std 1.0  -> factor 256 per layer                    -> explodes
# Xavier uses variance 1/WIDTH (factor 1, suits tanh). He uses 2/WIDTH because
# ReLU zeroes about half the units, halving the variance, so the 2 puts it back.
def run(init, act="relu", batchnorm=False, residual=False):
    # Ws stores each weight matrix; caches store what the backward pass needs.
    Ws, caches = [], []
    x = x0
    for layer in range(DEPTH):
        # Draw this layer's (256, 256) weight matrix using the chosen scheme.
        if init == "small":
            W = rng.normal(0, 0.01, (WIDTH, WIDTH))     # naive tiny weights
        elif init == "large":
            W = rng.normal(0, 1.0, (WIDTH, WIDTH))      # naive large weights
        elif init == "xavier":
            # Xavier/Glorot: variance 1/fan_in keeps the signal size constant.
            W = rng.normal(0, np.sqrt(1 / WIDTH), (WIDTH, WIDTH))
        else:                                           # he
            # He: variance 2/fan_in compensates for ReLU discarding half the signal.
            W = rng.normal(0, np.sqrt(2 / WIDTH), (WIDTH, WIDTH))
        # Linear step: (512, 256) @ (256, 256) -> (512, 256) pre-activations.
        z = x @ W
        # Batch normalisation forces every unit back to mean 0 and std 1, so
        # whatever the weights did, the next layer receives a well-scaled signal.
        # standardise each unit over the batch
        if batchnorm:
            z = (z - z.mean(0)) / (z.std(0) + 1e-5)
        h = np.maximum(z, 0) if act == "relu" else np.tanh(z)
        # Residual connection: output = input + small change. The identity path
        # carries the signal (and later the gradient) through untouched.
        if residual:
            # skip connection: add input back (small branch scale)
            h = x + h * 0.1
        # Remember W, z and the layer input for the backward pass.
        Ws.append(W); caches.append((z, x))
        x = h
    out_std = x.std()
 
    if batchnorm:
        # batch norm backward pass omitted to keep the code short
        return out_std, None
    # Backward: send a random gradient down through every layer (chain rule)
    # Each step multiplies the gradient by the activation's slope and by W.T,
    # so the same compounding that hits activations also hits gradients.
    g = rng.normal(0, 1, x.shape)
    for W, (z, x_in) in zip(reversed(Ws), reversed(caches)):
        # Slope of the activation: ReLU passes 1 or 0; tanh passes 1 - tanh^2,
        # which is near 0 when tanh saturates.
        d_act = (z > 0) if act == "relu" else 1 - np.tanh(z) ** 2
        # Gradient through the (possibly scaled) branch: (512, 256) @ W.T.
        branch = (g * (0.1 if residual else 1)) * d_act @ W.T
        # With a skip connection the gradient also flows straight through the
        # "+ x" path, so it can never be multiplied down to zero.
        g = g + branch if residual else branch
    return out_std, np.abs(g).mean()
 
 
# ---- Compare the seven configurations ----------------------------------------------
print(f"{'Configuration':40s} {'activation std at layer 50':>27s} "
      f"{'gradient size at layer 1':>25s}")
for label, kw in [
    ("Tiny random weights (std 0.01), ReLU", dict(init="small")),
    ("Large random weights (std 1.0), tanh", dict(init="large", act="tanh")),
    ("Large random weights (std 1.0), ReLU", dict(init="large")),
    ("Xavier init, tanh", dict(init="xavier", act="tanh")),
    ("He init, ReLU", dict(init="he")),
    ("Tiny weights + batch normalisation", dict(init="small", batchnorm=True)),
    ("He init + residual connections", dict(init="he", residual=True)),
]:
    # Silence overflow warnings: the exploding cases are meant to overflow.
    with np.errstate(all="ignore"):
        s, g = run(**kw)
    g_text = "not measured" if g is None else f"{g:.3e}"
    print(f"{label:40s} {s:27.3e} {g_text:>25s}")
 
print("\nHealthy training needs both numbers to stay roughly in the range 0.1 to 10.")

Configuration                             activation std at layer 50  gradient size at layer 1


Tiny random weights (std 0.01), ReLU                       5.936e-48                 4.551e-48


Large random weights (std 1.0), tanh                       9.738e-01                 8.039e+22


Large random weights (std 1.0), ReLU                       2.327e+52                 2.656e+52


Xavier init, tanh                                          9.320e-02                 8.622e-02


He init, ReLU                                              2.269e+00                 9.526e-01


Tiny weights + batch normalisation                         6.218e-01              not measured


He init + residual connections                             2.614e+00                 1.190e+00

Healthy training needs both numbers to stay roughly in the range 0.1 to 10.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Configuration                             activation std at layer 50  gradient size at layer 1
Tiny random weights (std 0.01), ReLU                       5.936e-48                 4.551e-48
Large random weights (std 1.0), tanh                       9.738e-01                 8.039e+22
Large random weights (std 1.0), ReLU                       2.327e+52                 2.656e+52
Xavier init, tanh                                          9.320e-02                 8.622e-02
He init, ReLU                                              2.269e+00                 9.526e-01
Tiny weights + batch normalisation                         6.218e-01              not measured
He init + residual connections                             2.614e+00                 1.190e+00
 
Healthy training needs both numbers to stay roughly in the range 0.1 to 10.
```

**What the output shows**

The numbers make the problem unmistakable. With tiny weights the gradient reaching the first layer is around 10 to the power of minus 48, far too small for any learning. With large weights activations and gradients explode to around 10 to the power of 52. Matching the initialisation to the activation function, adding batch normalisation, or using residual connections all keep signals in a usable range.

**Investigate**

1. Increase DEPTH to 200. Which configurations remain healthy?
2. Use He initialisation with tanh, and Xavier initialisation with ReLU. Explain the results using the variance argument from Section 4.10.
3. Change the residual branch scale from 0.1 to 1.0. What happens to the activation size, and why do real ResNets place normalisation inside each block?

## 2. Convolution from scratch

**File:** `l4_convolution.py`

In [3]:
"""
Layer 4 example: how convolution works, written from scratch in NumPy.
 
File: l4_convolution.py
 
Purpose:
    Convolution is the core operation of a convolutional neural network (CNN).
    A convolution slides a small grid of weights (a FILTER or KERNEL) across an
    image. At each position it multiplies overlapping values and adds them up,
    producing one number in the output FEATURE MAP. Because the same few
    weights are reused everywhere, a CNN needs far fewer parameters than a
    dense layer and detects a pattern wherever it appears.
 
How it works:
    1. Build a 7x7 image of a bright square on a dark background.
    2. Apply three hand-made filters (vertical edge, horizontal edge, blur).
    3. Show how padding and stride change the output size.
    4. Check translation equivariance and count parameters.
 
    Output size formula for an input of width W, filter size F, padding P and
    stride S:
        output width = (W - F + 2P) / S + 1
 
What to look for in the output:
    - The vertical edge filter gives +36 on the left edge of the square and
      -36 on the right edge, and 0 in the flat middle.
    - padding=1 stride=1 keeps the size at 7x7; stride 2 shrinks it to 3x3.
    - A dense layer on a 224x224 RGB image needs 9,633,856 parameters; a conv
      layer with 64 3x3 filters needs 1,792, i.e. 5,376 times fewer.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
 
# ---- conv2d(): the sliding-window operation -----------------------------------------
# For every output position (i, j) we cut out a kh x kw patch of the image and
# take its dot product with the kernel. The stride is how many pixels the
# window jumps each step; padding adds zeros around the border so edge pixels
# can sit at the centre of a window.
def conv2d(image, kernel, stride=1, padding=0):
    """Single-channel 2D convolution (strictly cross-correlation, as used in
    deep learning)."""
    if padding:
        image = np.pad(image, padding)                 # add a border of zeros
    kh, kw = kernel.shape
    # Output size formula (W - F + 2P) // S + 1; padding is already in image.
    out_h = (image.shape[0] - kh) // stride + 1
    out_w = (image.shape[1] - kw) // stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            # the 'receptive field'
            patch = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            # dot product: same kernel weights at every position (weight sharing)
            out[i, j] = np.sum(patch * kernel)
    return out
 
 
# Helper: print a matrix as a grid of whole numbers.
def show(name, m):
    print(f"{name} (shape {m.shape}):")
    for row in m:
        print("  " + " ".join(f"{v:4.0f}" for v in row))
 
 
# ---- A 7x7 test image: a bright square on a dark background -------------------
img = np.zeros((7, 7))
# Rows and columns 2..4 are bright (value 9); everything else is 0.
img[2:5, 2:5] = 9
show("Input image", img)
 
# ---- Hand-designed filters (a CNN LEARNS filters like these) ---------------------
# Sobel x: negative on the left, positive on the right -> responds to a
# left-right change in brightness (a vertical edge). Sobel y is the same
# turned 90 degrees. The box filter sums a 3x3 neighbourhood, i.e. blurs.
filters = {
    "Vertical edge (Sobel x)":   np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Horizontal edge (Sobel y)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Blur (box average x9)":     np.ones((3, 3)),
}
for name, k in filters.items():
    show("\n" + name, conv2d(img, k))
# Edge filters give large positive or negative values only where brightness changes.
 
# ---- Padding and stride change the output size ------------------------------------
# Compare the real output shape with the formula (W - F + 2P) // S + 1.
k = filters["Vertical edge (Sobel x)"]
print("\nOutput sizes for a 7x7 input and 3x3 filter:")
for p, s in [(0, 1), (1, 1), (0, 2), (1, 2)]:
    out = conv2d(img, k, stride=s, padding=p)
    formula = (7 - 3 + 2 * p) // s + 1
    print(f"  padding={p} stride={s}: output {out.shape}   "
          f"formula gives {formula}x{formula}")
 
# ---- Translation equivariance: shift the input, the feature map shifts too ------
# Because the same filter is used at every position, moving an object moves
# its response by the same amount. This is why CNNs find objects anywhere.
shifted = np.roll(img, 1, axis=1)                      # move square one pixel right
a, b = conv2d(img, k), conv2d(shifted, k)
# Compare a's columns 0..3 with b's columns 1..4 (b is a shifted right by 1).
print("\nShift input right by 1 -> feature map also shifts right by 1:",
      np.allclose(a[:, :-1], b[:, 1:]))
 
# ---- Why convolution? Parameter counts for a 224x224 RGB image ----------------
# A dense layer connects every input value to every neuron. A conv layer only
# stores its small filters, which are shared across all positions, so its
# parameter count does not depend on the image size at all.
inputs = 224 * 224 * 3
dense_params = inputs * 64 + 64                         # dense layer with 64 neurons
# 64 filters of size 3x3 over 3 channels
conv_params = 3 * 3 * 3 * 64 + 64
print(f"\nDense layer, 64 neurons: {dense_params:,} parameters")
print(f"Conv layer, 64 3x3 filters: {conv_params:,} parameters")
print(f"The convolutional layer needs {dense_params // conv_params:,} "
      f"times fewer parameters.")

Input image (shape (7, 7)):
     0    0    0    0    0    0    0
     0    0    0    0    0    0    0
     0    0    9    9    9    0    0
     0    0    9    9    9    0    0
     0    0    9    9    9    0    0
     0    0    0    0    0    0    0
     0    0    0    0    0    0    0

Vertical edge (Sobel x) (shape (5, 5)):
     9    9    0   -9   -9
    27   27    0  -27  -27
    36   36    0  -36  -36
    27   27    0  -27  -27
     9    9    0   -9   -9

Horizontal edge (Sobel y) (shape (5, 5)):
     9   27   36   27    9
     9   27   36   27    9
     0    0    0    0    0
    -9  -27  -36  -27   -9
    -9  -27  -36  -27   -9

Blur (box average x9) (shape (5, 5)):
     9   18   27   18    9
    18   36   54   36   18
    27   54   81   54   27
    18   36   54   36   18
     9   18   27   18    9

Output sizes for a 7x7 input and 3x3 filter:
  padding=0 stride=1: output (5, 5)   formula gives 5x5
  padding=1 stride=1: output (7, 7)   formula gives 7x7
  padding=0 stride=2: outpu

**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Input image (shape (7, 7)):
     0    0    0    0    0    0    0
     0    0    0    0    0    0    0
     0    0    9    9    9    0    0
     0    0    9    9    9    0    0
     0    0    9    9    9    0    0
     0    0    0    0    0    0    0
     0    0    0    0    0    0    0
 
Vertical edge (Sobel x) (shape (5, 5)):
     9    9    0   -9   -9
    27   27    0  -27  -27
    36   36    0  -36  -36
    27   27    0  -27  -27
     9    9    0   -9   -9
 
Horizontal edge (Sobel y) (shape (5, 5)):
     9   27   36   27    9
     9   27   36   27    9
     0    0    0    0    0
    -9  -27  -36  -27   -9
    -9  -27  -36  -27   -9
 
Blur (box average x9) (shape (5, 5)):
     9   18   27   18    9
    18   36   54   36   18
    27   54   81   54   27
    18   36   54   36   18
     9   18   27   18    9
 
Output sizes for a 7x7 input and 3x3 filter:
  padding=0 stride=1: output (5, 5)   formula gives 5x5
  padding=1 stride=1: output (7, 7)   formula gives 7x7
  padding=0 stride=2: output (3, 3)   formula gives 3x3
  padding=1 stride=2: output (4, 4)   formula gives 4x4
 
Shift input right by 1 -> feature map also shifts right by 1: True
 
Dense layer, 64 neurons: 9,633,856 parameters
Conv layer, 64 3x3 filters: 1,792 parameters
The convolutional layer needs 5,376 times fewer parameters.
```

**What the output shows**

The edge filters respond only where brightness changes, with the sign showing the direction of the change, while the blur filter spreads values smoothly. The output size formula matches every case, the feature map shifts exactly when the input shifts, and the parameter comparison shows why fully connected layers are impractical for images.

**Investigate**

1. Design a 3 by 3 filter that detects diagonal edges running from top left to bottom right, and test it on the square.
2. Implement a 2D convolution for a 3-channel input with a 3 by 3 by 3 filter.
3. The Sobel filters are hand-designed. In a CNN, what process sets the filter values?

## 3. A CNN trained from scratch in NumPy

**File:** `l4_cnn_numpy.py`

In [4]:
"""
Layer 4 example: a complete convolutional neural network trained in NumPy.
 
File: l4_cnn_numpy.py
 
Purpose:
    Shows every piece of a CNN, forward and backward, with no framework:
    convolution, ReLU, max pooling, a dense layer and softmax, trained by
    gradient descent on 8x8 handwritten digits. It also shows why weight
    sharing lets a small CNN match a dense network with more parameters.
 
How it works:
    Architecture for 8x8 greyscale digits:
      input 1x8x8 -> CONV 8 filters 3x3 -> ReLU -> MAX POOL 2x2
        -> flatten 72 -> DENSE 10 -> softmax
 
    The 'im2col' trick turns every 3x3 patch into a row of a matrix, so the
    whole convolution becomes one fast matrix multiplication. Frameworks use
    the same idea (plus highly optimised GPU kernels).
    The network trains for 40 epochs of mini-batches of 32, then is compared
    with a small dense MLP, and its learned filters are printed.
 
What to look for in the output:
    - Test accuracy rises from 0.264 after epoch 1 to 0.971 after epoch 40.
      Epoch 20 dips to 0.616: plain SGD with a fixed learning rate can bounce.
    - The CNN has 810 parameters; a dense MLP with 910 reaches 0.967.
    - The conv layer's 80 parameters are reused at 36 positions, giving 288
      feature detectors.
    - The learned filters show +/- patterns, i.e. simple edge detectors.
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import warnings
# Hide the MLP's convergence warning to keep the output clean.
warnings.filterwarnings("ignore")
 
rng = np.random.default_rng(0)
 
# ---- Data --------------------------------------------------------------------------
# 1,797 digit images; pixel values 0..16 are scaled to 0..1.
X, y = load_digits(return_X_y=True)
X = (X / 16.0).reshape(-1, 1, 8, 8)                    # (N, channels, height, width)
# Hold back 25% for testing; stratify keeps each digit's share the same.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0,
                                          stratify=y)
 
 
# ---- im2col: turn convolution into a matrix multiplication -----------------------
# A 3x3 filter on an 8x8 image fits at (8 - 3 + 1)^2 = 6x6 = 36 positions.
# im2col copies each 3x3 patch into a row of 9 numbers. Then applying all
# filters at all positions is one product: (N, 36, 9) @ (9, F) -> (N, 36, F).
def im2col(x, k=3):
    """(N, 1, 8, 8) -> (N, 36, 9): 36 patch positions, 9 pixels per patch."""
    N, _, H, W = x.shape
    out_h, out_w = H - k + 1, W - k + 1
    cols = np.empty((N, out_h * out_w, k * k))
    for i in range(out_h):
        for j in range(out_w):
            # Patch at (i, j) flattened to 9 values, stored as row i*out_w + j.
            cols[:, i * out_w + j, :] = x[:, 0, i:i+k, j:j+k].reshape(N, -1)
    return cols
 
 
# ---- Parameters ------------------------------------------------------------------
# He initialisation (std sqrt(2 / fan_in)) suits the ReLU that follows.
F = 8                                                   # number of filters
# each filter: 9 weights (He init)
K = rng.normal(0, np.sqrt(2 / 9), (F, 9))
bK = np.zeros(F)
# After pooling: 3 x 3 positions x 8 filters = 72 features into 10 classes.
Wd = rng.normal(0, np.sqrt(2 / 72), (72, 10))           # dense layer after pooling
bd = np.zeros(10)
 
 
# ---- Forward pass ----------------------------------------------------------------
# Returns the class scores (logits) plus every intermediate array, which the
# backward pass needs in order to apply the chain rule.
def forward(x):
    cols = im2col(x)                                    # (N, 36, 9)
    # (N, 36, 8): every filter at every position
    conv = cols @ K.T + bK
    # Rearrange the 36 positions back into a 6x6 grid: (N, 6, 6, 8).
    conv = conv.reshape(-1, 6, 6, F)
    relu = np.maximum(conv, 0)
    # 2x2 max pooling: 6x6 -> 3x3, keep the largest value in each block
    # Reshape splits each 6-axis into (3 blocks, 2 within block):
    # (N, 3, 2, 3, 2, 8); taking max over the "within" axes gives (N, 3, 3, 8).
    blocks = relu.reshape(-1, 3, 2, 3, 2, F)
    pooled = blocks.max(axis=(2, 4))                    # (N, 3, 3, 8)
    flat = pooled.reshape(len(x), -1)                   # (N, 72)
    logits = flat @ Wd + bd                             # (N, 10) class scores
    return logits, (cols, conv, relu, blocks, pooled, flat)
 
 
# ---- One training step: forward, backward, update --------------------------------
# Backpropagation runs the network in reverse, turning "how wrong was the
# output" into a gradient for every weight.
def train_step(x, labels, lr):
    global K, bK, Wd, bd
    N = len(x)
    logits, (cols, conv, relu, blocks, pooled, flat) = forward(x)
    # softmax + cross-entropy
    # Subtracting the row max before exp avoids overflow without changing p.
    p = np.exp(logits - logits.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    loss = -np.log(p[np.arange(N), labels] + 1e-12).mean()
    # Gradient of softmax + cross-entropy is simply (p - one_hot) / N.
    d_logits = p.copy(); d_logits[np.arange(N), labels] -= 1; d_logits /= N
 
    # dense layer gradients
    dWd = flat.T @ d_logits; dbd = d_logits.sum(0)
    # Send the gradient back to the pooled features: (N, 72) -> (N, 3, 3, 8).
    d_pooled = (d_logits @ Wd.T).reshape(pooled.shape)
 
    # max pool backward: gradient goes ONLY to the position that was the maximum
    # The other three values in each 2x2 block had no effect on the output,
    # so their gradient is zero. mask marks the winners.
    mask = (blocks == pooled[:, :, None, :, None, :])
    d_relu = (mask * d_pooled[:, :, None, :, None, :]).reshape(relu.shape)
 
    d_conv = d_relu * (conv > 0)                        # ReLU backward
    d_conv = d_conv.reshape(N, 36, F)
    # Weight sharing in the backward pass: each filter's gradient is the SUM
    # of its contributions from all 36 positions and all N images.
    dK = np.einsum("npf,npk->fk", d_conv, cols)         # sum over images and positions
    dbK = d_conv.sum(axis=(0, 1))
 
    # Gradient descent: move every parameter a small step downhill.
    K -= lr * dK; bK -= lr * dbK; Wd -= lr * dWd; bd -= lr * dbd
    return loss
 
 
# ---- Training loop -------------------------------------------------------------------
# Each epoch shuffles the training set and walks through it in batches of 32.
print("Training the NumPy CNN:")
for epoch in range(1, 41):
    order = rng.permutation(len(X_tr))
    for s in range(0, len(order), 32):
        idx = order[s:s+32]
        loss = train_step(X_tr[idx], y_tr[idx], lr=0.05)
    if epoch in (1, 10, 20, 40):
        # Accuracy: fraction of test images whose highest score is the true class.
        acc = (forward(X_te)[0].argmax(1) == y_te).mean()
        # Cross-entropy on the whole training set (softmax written out in full).
        full_loss = -np.log(
            np.exp(forward(X_tr)[0] - forward(X_tr)[0].max(1, keepdims=True))[
                np.arange(len(y_tr)), y_tr]
            / np.exp(forward(X_tr)[0] - forward(X_tr)[0].max(1, keepdims=True)).sum(1)
        ).mean()
        print(f"  epoch {epoch:2d}  training loss {full_loss:.3f}  "
              f"test accuracy {acc:.3f}")
 
# 8*9 filter weights + 8 biases + 72*10 dense weights + 10 biases = 810.
cnn_params = K.size + bK.size + Wd.size + bd.size
print(f"CNN parameters: {cnn_params}")
 
# ---- Compare with a dense network of similar size ---------------------------------
# 64 inputs -> 12 hidden -> 10 outputs gives 910 parameters, close to the CNN.
mlp = MLPClassifier(hidden_layer_sizes=(12,), max_iter=300, random_state=0).fit(
    X_tr.reshape(len(X_tr), -1), y_tr)
mlp_params = sum(w.size for w in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print(f"Dense MLP parameters: {mlp_params}   "
      f"test accuracy {mlp.score(X_te.reshape(len(X_te), -1), y_te):.3f}")
 
# ---- Weight sharing in numbers -------------------------------------------------------
# The 8 conv filters (80 parameters) are applied at 36 positions each, so this
# layer performs 288 'neuron' computations with only 80 weights.
print(f"Conv layer: {K.size + bK.size} parameters reused at 36 positions = "
      f"{F * 36} feature detectors")
 
# ---- Look at what the filters learned --------------------------------------------
# Each filter is shown row by row; values beyond 35% of its largest weight
# count as clearly positive or negative.
print("\nLearned 3x3 filters ('+' positive, '-' negative, '.' near zero):")
for f in range(4):
    k = K[f].reshape(3, 3)
    t = np.abs(k).max() * 0.35
    rows = ["".join("+" if v > t else "-" if v < -t else "." for v in r) for r in k]
    print(f"  filter {f}: " + " | ".join(rows))

Training the NumPy CNN:
  epoch  1  training loss 1.934  test accuracy 0.264


  epoch 10  training loss 0.302  test accuracy 0.918


  epoch 20  training loss 1.029  test accuracy 0.616


  epoch 40  training loss 0.103  test accuracy 0.971
CNN parameters: 810


Dense MLP parameters: 910   test accuracy 0.967
Conv layer: 80 parameters reused at 36 positions = 288 feature detectors

Learned 3x3 filters ('+' positive, '-' negative, '.' near zero):
  filter 0: ..+ | ... | ++-
  filter 1: -.. | -.- | ...
  filter 2: ++. | +.. | .--
  filter 3: --. | -.. | ++.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Training the NumPy CNN:
  epoch  1  training loss 1.934  test accuracy 0.264
  epoch 10  training loss 0.302  test accuracy 0.918
  epoch 20  training loss 1.029  test accuracy 0.616
  epoch 40  training loss 0.103  test accuracy 0.971
CNN parameters: 810
Dense MLP parameters: 910   test accuracy 0.967
Conv layer: 80 parameters reused at 36 positions = 288 feature detectors
 
Learned 3x3 filters ('+' positive, '-' negative, '.' near zero):
  filter 0: ..+ | ... | ++-
  filter 1: -.. | -.- | ...
  filter 2: ++. | +.. | .--
  filter 3: --. | -.. | ++.
```

**What the output shows**

The CNN reaches about 97 percent accuracy with 810 parameters. Note the temporary collapse at epoch 20: with plain SGD and a fairly high learning rate, a few large updates can knock training off course before it recovers, which is exactly why frameworks default to adaptive optimisers and why practitioners monitor training curves. On tiny 8 by 8 images a small dense network performs just as well; the advantages of convolution grow with image size, as the parameter calculation in the previous example showed.

**Investigate**

1. Add momentum or Adam to train_step() and check whether the instability at epoch 20 disappears.
2. Increase the number of filters to 16. How do the parameter count and accuracy change?
3. Print the 6 by 6 feature map produced by one filter for a test image of a zero and describe what the filter responds to.

## 4. A character-level RNN

**File:** `l4_rnn.py`

Andrej Karpathy's widely read 2015 article showed how character-level RNNs trained on text learn spelling, punctuation and even the structure of source code (Karpathy, 2015). The program below implements a small character-level RNN with BPTT, gradient clipping and the Adagrad optimiser.

In [5]:
"""
Layer 4 example: a recurrent neural network (RNN) trained with
backpropagation through time (BPTT), written in NumPy.
 
File: l4_rnn.py
 
Purpose:
    An RNN reads a sequence one step at a time and carries a hidden state
    forward as memory, so it can model text, speech and time series. This
    program trains one to predict the next character, and then measures how
    quickly the training signal fades as it travels back in time, which is
    the vanishing-gradient problem that motivated the LSTM.
 
    Task: character-level language modelling. Given the characters so far,
    predict the next character. The hidden state h acts as a running memory:
        h_t = tanh(W_xh x_t + W_hh h_(t-1) + b_h)
        y_t = W_hy h_t + b_y        (scores for the next character)
    The SAME weights are reused at every time step.
 
How it works:
    1. Cut the text into chunks of 25 characters (inputs) and the same chunk
       shifted by one (targets).
    2. Unroll the RNN over the 25 steps, compute the loss, then run BPTT to
       get gradients, clip them, and update with Adagrad.
    3. Every so often, generate a 45-character sample from the model.
    4. Finally, measure the gradient size 0 to 39 steps back in time.
 
What to look for in the output:
    - Loss falls from 2.639 at iteration 1 to 0.030 at iteration 3000.
    - The sample goes from gibberish to 'the cat sat on the log. the cat
      sat on the mat'.
    - The gradient shrinks from 1.47e-01 at 0 steps back to 1.31e-07 at 39
      steps back: the RNN barely learns from events far in the past.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
rng = np.random.default_rng(1)
# A small, repetitive training text so the network can learn it quickly.
text = "the cat sat on the mat. the dog sat on the log. " * 20
chars = sorted(set(text))
V = len(chars)                                  # vocabulary size
# Map each character to an integer index.
c2i = {c: i for i, c in enumerate(chars)}
H = 48                                          # hidden units
SEQ = 25                                        # unroll 25 steps for each update
 
# ---- Parameters ----------------------------------------------------------------------
# Wxh: input -> hidden (H, V); Whh: hidden -> hidden (H, H), the recurrent
# weights that carry memory; Why: hidden -> output scores (V, H).
Wxh = rng.normal(0, 0.01, (H, V)); Whh = rng.normal(0, 0.01, (H, H))
Why = rng.normal(0, 0.01, (V, H))
bh = np.zeros(H); by = np.zeros(V)
params = [Wxh, Whh, Why, bh, by]
# Adagrad keeps a running sum of squared gradients per weight and divides the
# step by its square root, so often-updated weights take smaller steps.
# Adagrad memory (adaptive learning rate)
mem = [np.zeros_like(p) for p in params]
 
 
# Turn a character index into a length-V vector with a single 1.
def one_hot(i):
    v = np.zeros(V); v[i] = 1; return v
 
 
# ---- step(): forward over a chunk, then backpropagation through time -------------
# BPTT treats the unrolled RNN as one deep network with a layer per time step,
# where every "layer" shares the same weights. Gradients from each step are
# added together into the shared weights.
def step(inputs, targets, h_prev):
    # Dictionaries keyed by time step; hs[-1] is the memory from the last chunk.
    xs, hs, ps = {}, {-1: h_prev}, {}
    loss = 0.0
    # ---- forward through time ----
    for t, (i, j) in enumerate(zip(inputs, targets)):
        xs[t] = one_hot(i)
        # New memory mixes the current character with the previous memory.
        hs[t] = np.tanh(Wxh @ xs[t] + Whh @ hs[t-1] + bh)
        s = Why @ hs[t] + by
        # Softmax turns scores into next-character probabilities.
        ps[t] = np.exp(s - s.max()); ps[t] /= ps[t].sum()
        # Cross-entropy: penalise low probability for the true next character.
        loss -= np.log(ps[t][j])
    # ---- backward through time (BPTT) ----
    grads = [np.zeros_like(p) for p in params]
    dWxh, dWhh, dWhy, dbh, dby = grads
    # dh_next carries gradient from step t+1 back into step t.
    dh_next = np.zeros(H)
    for t in reversed(range(len(inputs))):
        # softmax + cross-entropy gradient
        dy = ps[t].copy(); dy[targets[t]] -= 1
        dWhy += np.outer(dy, hs[t]); dby += dy
        # gradient from output AND from the future
        dh = Why.T @ dy + dh_next
        draw = (1 - hs[t] ** 2) * dh                    # through tanh
        dbh += draw; dWxh += np.outer(draw, xs[t]); dWhh += np.outer(draw, hs[t-1])
        # Each step back multiplies by Whh.T and a tanh slope (at most 1), so
        # the gradient tends to shrink (vanish) or, with large Whh, explode.
        # carried back to the previous time step
        dh_next = Whh.T @ draw
    # Clipping caps every gradient value at +/-5, so one bad chunk cannot
    # throw the weights far away. It fixes exploding, not vanishing, gradients.
    for g in grads:
        # GRADIENT CLIPPING stops explosions
        np.clip(g, -5, 5, out=g)
    # Return the average loss, the gradients and the final memory for the
    # next chunk.
    return loss / len(inputs), grads, hs[len(inputs) - 1]
 
 
# ---- sample(): let the model write text ---------------------------------------------
# Feed each generated character back in as the next input, drawing the next
# character at random according to the predicted probabilities.
def sample(seed_char, n):
    h, i, out = np.zeros(H), c2i[seed_char], seed_char
    for _ in range(n):
        h = np.tanh(Wxh @ one_hot(i) + Whh @ h + bh)
        s = Why @ h + by
        p = np.exp(s - s.max()); p /= p.sum()
        i = rng.choice(V, p=p)
        out += chars[i]
    return out
 
 
# ---- Training loop -------------------------------------------------------------------
# pos walks through the text; h carries memory from one chunk to the next.
pos, h = 0, np.zeros(H)
for it in range(1, 3001):
    # Wrap to the start (and reset memory) when the text runs out.
    if pos + SEQ + 1 > len(text):
        pos, h = 0, np.zeros(H)
    # Targets are the inputs shifted one character to the right.
    inp = [c2i[c] for c in text[pos:pos+SEQ]]
    tgt = [c2i[c] for c in text[pos+1:pos+SEQ+1]]
    loss, grads, h = step(inp, tgt, h)
    for p, g, m in zip(params, grads, mem):
        m += g * g
        p -= 0.1 * g / np.sqrt(m + 1e-8)                # Adagrad update
    pos += SEQ
    if it in (1, 300, 1000, 3000):
        print(f"iteration {it:4d}  loss {loss:.3f}  sample: {sample('t', 45)!r}")
 
# ---- How far back does the gradient reach? -----------------------------------------
# Measure the size of dh_next as it is carried back through time from one output.
# Run the trained RNN forward over 40 characters and store every hidden state.
inp = [c2i[c] for c in text[:40]]
hs = {-1: np.zeros(H)}
for t, i in enumerate(inp):
    hs[t] = np.tanh(Wxh @ one_hot(i) + Whh @ hs[t-1] + bh)
# Start with a gradient of ones on the output at the last step.
dh = Why.T @ np.ones(V)
sizes = []
for t in reversed(range(40)):
    # Same two multiplications as in BPTT: tanh slope, then Whh.T.
    draw = (1 - hs[t] ** 2) * dh
    sizes.append(np.abs(draw).mean())
    dh = Whh.T @ draw
print("\nGradient size flowing back from step 40 to earlier steps:")
for back in (0, 5, 10, 20, 39):
    print(f"  {back:2d} steps back: {sizes[back]:.2e}")

iteration    1  loss 2.639  sample: 'ttlcccah.lesscaegmtsehhs.oc.tdnechac mt csh.ca'


iteration  300  loss 0.047  sample: 't e dog sat on the mat. the dog sat on the log'


iteration 1000  loss 0.034  sample: 'the cat sat on the mat. the dog sat on the mat'


iteration 3000  loss 0.030  sample: 'the cat sat on the log. the cat sat on the mat'

Gradient size flowing back from step 40 to earlier steps:
   0 steps back: 1.47e-01
   5 steps back: 1.28e-02
  10 steps back: 5.95e-03
  20 steps back: 1.64e-04
  39 steps back: 1.31e-07


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
iteration    1  loss 2.639  sample: 'ttlcccah.lesscaegmtsehhs.oc.tdnechac mt csh.ca'
iteration  300  loss 0.047  sample: 't e dog sat on the mat. the dog sat on the log'
iteration 1000  loss 0.034  sample: 'the cat sat on the mat. the dog sat on the mat'
iteration 3000  loss 0.030  sample: 'the cat sat on the log. the cat sat on the mat'
 
Gradient size flowing back from step 40 to earlier steps:
   0 steps back: 1.47e-01
   5 steps back: 1.28e-02
  10 steps back: 5.95e-03
  20 steps back: 1.64e-04
  39 steps back: 1.31e-07
```

**What the output shows**

Within a few hundred iterations the network generates the training phrases almost perfectly, having learned spelling and word order one character at a time. The final measurement shows the vanishing gradient directly: the learning signal from one output is about a million times weaker 39 steps back than at the step itself, so the RNN cannot easily learn dependencies over long distances.

**Investigate**

1. Train on a paragraph of real English text of your choice. How does the sample quality change with 1,000 and 10,000 iterations?
2. Remove the gradient clipping line. Does training still succeed?
3. Increase SEQ to 100. What happens to training speed and loss?

## 5. How gates preserve memory

**File:** `l4_lstm.py`

In [6]:
"""
Layer 4 example: inside an LSTM cell (Hochreiter and Schmidhuber, 1997).
 
File: l4_lstm.py
 
Purpose:
    Plain RNNs forget quickly because their memory is squashed and multiplied
    at every step. The Long Short-Term Memory (LSTM) cell adds a protected
    memory lane with gates that decide what to keep, write and reveal, so it
    can remember information, and pass gradients, over many time steps.
 
    An LSTM keeps a separate CELL STATE c that runs along the sequence with
    only small, controlled changes. Three GATES (values 0 to 1) decide:
      forget gate f: how much of the old memory to keep
      input gate  i: how much new information to write
      output gate o: how much of the memory to reveal as the hidden state h
 
        f = sigmoid(Wf [h, x] + bf)       i = sigmoid(Wi [h, x] + bi)
        g = tanh(Wg [h, x] + bg)          o = sigmoid(Wo [h, x] + bo)
        c_new = f * c + i * g             h_new = o * tanh(c_new)
 
    Because c_new = f * c + ..., the gradient along the cell state is
    multiplied by f at each step. If f is near 1, information and gradients
    survive for many steps. A plain RNN multiplies by W_hh and a tanh
    derivative instead.
 
How it works:
    1. A hand-set one-unit LSTM sees a single 1 followed by 50 zeros; we
       check how much of it remains in memory for three forget gate biases.
    2. A plain RNN unit tries the same job with three recurrent weights.
    3. We compare how much gradient survives 50 steps in each.
 
What to look for in the output:
    - Forget bias 0 (f = 0.5): the cell is 0.0010 after 10 steps, gone.
    - Forget bias 8 (f = 0.9997): the cell is still 0.9768 after 50 steps.
    - The best plain RNN (weight 1.0) keeps only 0.1698 at step 50.
    - Gradient after 50 steps: LSTM with bias 8.0 keeps 9.834e-01, the plain
      RNN only 8.061e-04.
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
# Sigmoid squashes any number into 0..1, which is why gates use it: 0 means
# "closed", 1 means "fully open".
sig = lambda z: 1 / (1 + np.exp(-z))
 
# ---- A hand-set one-unit LSTM that REMEMBERS a signal ------------------------------
# Input sequence: a single '1' at step 0 followed by zeros. Can it remember at step 50?
# Weights chosen by hand for clarity: gates depend on the input x only.
def lstm_step(x, h, c, forget_bias):
    f = sig(forget_bias)                  # forget gate, constant here
    i = sig(10 * x - 5)                   # open only when x = 1
    g = np.tanh(10 * x)                   # candidate value to store
    o = sig(5.0)                          # output gate mostly open
    # The key line: keep a fraction f of the old memory, add the gated new value.
    # Addition (not repeated squashing) is what lets memory last.
    c = f * c + i * g
    # The hidden state is a gated, squashed view of the memory.
    h = o * np.tanh(c)
    return h, c, f, i
 
# One pulse at step 0, then 50 steps of silence.
seq = [1.0] + [0.0] * 50
# Try three forget gate biases: sigmoid(0) = 0.5, sigmoid(3) = 0.95,
# sigmoid(8) = 0.9997.
for fb in (0.0, 3.0, 8.0):
    h, c = 0.0, 0.0
    for t, x in enumerate(seq):
        h, c, f, i = lstm_step(x, h, c, fb)
        if t in (0, 10, 50):
            print(f"forget bias {fb:3.1f}  step {t:2d}: cell {c:.4f}  "
                  f"hidden {h:.4f}  forget gate {f:.4f}")
    print()
 
# ---- Plain RNN unit trying the same job -------------------------------------------
# Here memory is h itself, multiplied by w and squashed by tanh every step,
# so it decays unless w is large enough (and a large w risks explosion).
for w in (0.5, 0.9, 1.0):
    h = 0.0
    for t, x in enumerate(seq):
        h = np.tanh(w * h + 3 * x)
        if t in (0, 10, 50):
            print(f"RNN recurrent weight {w}: step {t:2d} hidden {h:.4f}")
    print()
 
# ---- Gradient survival over 50 steps -------------------------------------------------
# LSTM cell-state path: product of forget gates. RNN path: product of w * tanh'(.)
# LSTM: d c_50 / d c_0 = f^50, so a forget gate near 1 keeps it near 1.
for fb in (0.0, 3.0, 8.0):
    print(f"LSTM gradient after 50 steps with forget bias {fb}: {sig(fb) ** 50:.3e}")
# Plain RNN: each step multiplies by w * (1 - tanh^2), which is below 1.
h, prod = 0.9, 1.0
for t in range(50):
    z = 0.9 * h
    h = np.tanh(z)
    prod *= 0.9 * (1 - h ** 2)
print(f"Plain RNN gradient after 50 steps (w = 0.9):     {prod:.3e}")
print("\nThis is why LSTM implementations often initialise the forget gate "
      "bias to 1 or more.")

forget bias 0.0  step  0: cell 0.9933  hidden 0.7537  forget gate 0.5000
forget bias 0.0  step 10: cell 0.0010  hidden 0.0010  forget gate 0.5000
forget bias 0.0  step 50: cell 0.0000  hidden 0.0000  forget gate 0.5000

forget bias 3.0  step  0: cell 0.9933  hidden 0.7537  forget gate 0.9526
forget bias 3.0  step 10: cell 0.6110  hidden 0.5412  forget gate 0.9526
forget bias 3.0  step 50: cell 0.0875  hidden 0.0867  forget gate 0.9526

forget bias 8.0  step  0: cell 0.9933  hidden 0.7537  forget gate 0.9997
forget bias 8.0  step 10: cell 0.9900  hidden 0.7523  forget gate 0.9997
forget bias 8.0  step 50: cell 0.9768  hidden 0.7466  forget gate 0.9997

RNN recurrent weight 0.5: step  0 hidden 0.9951
RNN recurrent weight 0.5: step 10 hidden 0.0009
RNN recurrent weight 0.5: step 50 hidden 0.0000

RNN recurrent weight 0.9: step  0 hidden 0.9951
RNN recurrent weight 0.9: step 10 hidden 0.1842
RNN recurrent weight 0.9: step 50 hidden 0.0026

RNN recurrent weight 1.0: step  0 hidden 0.9951
RN

**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
forget bias 0.0  step  0: cell 0.9933  hidden 0.7537  forget gate 0.5000
forget bias 0.0  step 10: cell 0.0010  hidden 0.0010  forget gate 0.5000
forget bias 0.0  step 50: cell 0.0000  hidden 0.0000  forget gate 0.5000
 
forget bias 3.0  step  0: cell 0.9933  hidden 0.7537  forget gate 0.9526
forget bias 3.0  step 10: cell 0.6110  hidden 0.5412  forget gate 0.9526
forget bias 3.0  step 50: cell 0.0875  hidden 0.0867  forget gate 0.9526
 
forget bias 8.0  step  0: cell 0.9933  hidden 0.7537  forget gate 0.9997
forget bias 8.0  step 10: cell 0.9900  hidden 0.7523  forget gate 0.9997
forget bias 8.0  step 50: cell 0.9768  hidden 0.7466  forget gate 0.9997
 
RNN recurrent weight 0.5: step  0 hidden 0.9951
RNN recurrent weight 0.5: step 10 hidden 0.0009
RNN recurrent weight 0.5: step 50 hidden 0.0000
 
RNN recurrent weight 0.9: step  0 hidden 0.9951
RNN recurrent weight 0.9: step 10 hidden 0.1842
RNN recurrent weight 0.9: step 50 hidden 0.0026
 
RNN recurrent weight 1.0: step  0 hidden 0.9951
RNN recurrent weight 1.0: step 10 hidden 0.3560
RNN recurrent weight 1.0: step 50 hidden 0.1698
 
LSTM gradient after 50 steps with forget bias 0.0: 8.882e-16
LSTM gradient after 50 steps with forget bias 3.0: 8.809e-02
LSTM gradient after 50 steps with forget bias 8.0: 9.834e-01
Plain RNN gradient after 50 steps (w = 0.9):     8.061e-04
 
This is why LSTM implementations often initialise the forget gate bias to 1 or more.
```

**What the output shows**

With a forget gate near 1, the LSTM still holds 98 percent of the stored signal after 50 steps and passes back 98 percent of the gradient, whereas the plain RNN's memory and gradient fade rapidly.

**Investigate**

1. Modify the LSTM so the forget gate closes (output near 0) when a special input value of -1 appears, allowing the network to deliberately erase its memory.
2. Implement one step of a GRU cell and compare its equations with the LSTM.

## 6. Attention from scratch

**File:** `l4_attention.py`

In [7]:
"""
Layer 4 example: scaled dot-product self-attention and positional encoding,
the core components of the transformer (Vaswani et al., 2017).
 
File: l4_attention.py
 
Purpose:
    Attention lets every token in a sequence look directly at every other
    token and decide which ones matter, instead of passing memory step by
    step like an RNN. It is the building block of transformers and so of
    modern large language models.
 
        Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
 
    Every token produces a QUERY (what am I looking for?), a KEY (what do I
    offer?) and a VALUE (what information do I pass on?). Each token's output
    is a weighted average of all VALUES, weighted by how well its query
    matches each key. All tokens are processed IN PARALLEL, unlike an RNN.
 
How it works:
    1. Hand-built 4-dimensional word vectors show what 'it' attends to.
    2. Random projection matrices make queries, keys and values for two heads.
    3. A causal mask stops tokens looking at future tokens.
    4. An experiment shows why scores are divided by sqrt(d_k).
    5. Sinusoidal positional encodings give the model a sense of order.
 
What to look for in the output:
    - 'it' gives its largest weight to 'cat' (0.169).
    - The causal weight matrix has zeros everywhere above the diagonal.
    - At d_k=1024 the largest weight is 0.995 without scaling (nearly
      one-hot) but 0.281 with scaling.
    - Position 10 is most similar to its neighbours 9 and 11 (0.94) and less
      to position 30 (0.72); without positions, shuffling tokens just
      shuffles outputs (True).
 
Requires:
    pip install numpy
"""
 
import numpy as np
 
rng = np.random.default_rng(0)
 
 
# Softmax turns a row of scores into positive weights that sum to 1.
# Subtracting the max first avoids overflow in exp.
def softmax(z, axis=-1):
    e = np.exp(z - z.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)
 
 
# ---- attention(): the whole mechanism in four lines ---------------------------------
# Q is (n_tokens, d_k), K is (n_tokens, d_k), V is (n_tokens, d_v).
# scores[a, b] says how much token a's query matches token b's key.
# Dividing by sqrt(d_k) keeps the scores' spread about the same whatever the
# dimension, so softmax does not saturate.
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    # (n, n) matrix of dot products.
    # similarity of every query with every key
    scores = Q @ K.T / np.sqrt(d_k)
    # A score of -1e9 becomes a weight of 0 after softmax.
    if mask is not None:
        scores = np.where(mask, scores, -1e9)   # block disallowed positions
    weights = softmax(scores)                  # each row sums to 1
    # Each output row is a weighted average of the value vectors: (n, d_v).
    return weights @ V, weights
 
 
# ---- 1. A hand-built example you can reason about ------------------------------------
tokens = ["the", "cat", "chased", "the", "mouse", "because", "it", "was", "hungry"]
# Toy 4-dimensional embeddings: dimensions ~ [animal, predator, action, function word]
E = {"the": [0, 0, 0, 1], "cat": [1, 1, 0, 0], "chased": [0, 0.5, 1, 0],
     "mouse": [1, 0, 0, 0],
     "because": [0, 0, 0, 1], "it": [0.9, 0.6, 0, 0.2], "was": [0, 0, 0.3, 1],
     "hungry": [0.3, 0.8, 0.2, 0]}
# Stack into a (9 tokens, 4 dimensions) matrix.
X = np.array([E[t] for t in tokens], dtype=float)
 
# Use embeddings directly as queries, keys and values (no learned projections yet)
out, W = attention(X, X, X)
i = tokens.index("it")
print("Attention weights for the token 'it':")
# Row i of W is how 'it' spreads its attention; show the top four.
for t, w in sorted(zip(tokens, W[i]), key=lambda p: -p[1])[:4]:
    print(f"  {t:8s} {w:.3f}")
print("'it' attends most to the animal words, 'cat' in particular "
      "(a predator that is hungry).\n")
 
# ---- 2. Learned projections and multi-head attention --------------------------------
# Each head gets its own W_q, W_k, W_v, projecting 8 dimensions down to 4, so
# different heads can look for different relationships (e.g. grammar vs
# meaning). Their outputs are concatenated and mixed by W_o.
d_model, n_heads = 8, 2
d_head = d_model // n_heads
X8 = rng.normal(size=(len(tokens), d_model))   # (9, 8) random token vectors
heads = []
for h in range(n_heads):
    # In a real model W_q, W_k, W_v are LEARNED by backpropagation
    Wq, Wk, Wv = (rng.normal(0, 0.5, (d_model, d_head)) for _ in range(3))
    # Q, K and V are each (9, 4) for this head.
    o, w = attention(X8 @ Wq, X8 @ Wk, X8 @ Wv)
    heads.append(o)
    print(f"head {h}: token 'it' attends most strongly to '{tokens[w[i].argmax()]}'")
Wo = rng.normal(0, 0.5, (d_model, d_model))
# Two (9, 4) head outputs -> (9, 8), then mixed by W_o -> (9, 8).
multi = np.concatenate(heads, axis=1) @ Wo     # concatenate heads, then mix
print("multi-head output shape:", multi.shape, "(one vector per token)\n")
 
# ---- 3. Causal mask: a text generator must not look at future tokens -----------------
# When predicting word t, the model may only use words 0..t, so entries above
# the diagonal (future positions) are masked out.
n = 5
causal = np.tril(np.ones((n, n), dtype=bool))  # lower triangle = allowed
_, Wc = attention(X[:n], X[:n], X[:n], mask=causal)
print("Causal attention weights (rows = token, columns = attended token):")
print(np.round(Wc, 2))
print("Everything above the diagonal is zero: no token sees the future.\n")
 
# ---- 4. Why divide by sqrt(d_k)? -----------------------------------------------------
# A dot product of d random numbers has spread about sqrt(d). Large scores push
# softmax towards one-hot, where its gradients are almost zero. Dividing by
# sqrt(d) brings the spread back to about 1.
for d in (4, 64, 1024):
    # One query against ten keys.
    q, k = rng.normal(size=(1, d)), rng.normal(size=(10, d))
    raw = softmax(q @ k.T)[0].max()
    scaled = softmax(q @ k.T / np.sqrt(d))[0].max()
    print(f"d_k={d:5d}: largest weight without scaling {raw:.3f}, "
          f"with scaling {scaled:.3f}")
print("Without scaling, large dimensions make softmax nearly one-hot and "
      "gradients vanish.\n")
 
# ---- 5. Sinusoidal positional encoding -----------------------------------------------
# Each position gets a vector of sines and cosines at different frequencies,
# from fast (early dimensions) to slow (later ones), like the hands of a
# clock. It is added to the token embeddings so attention can use order.
def positional_encoding(n_pos, d):
    pos = np.arange(n_pos)[:, None]            # (n_pos, 1)
    i = np.arange(d)[None, :]                  # (1, d)
    # Dimension pairs (2k, 2k+1) share the frequency 1 / 10000^(2k/d).
    angle = pos / np.power(10000, (2 * (i // 2)) / d)
    # sin on even dimensions, cos on odd dimensions.
    return np.where(i % 2 == 0, np.sin(angle), np.cos(angle))
 
PE = positional_encoding(50, 16)
print("Positional encodings give each position a unique pattern.")
# Dividing by 8 (half of 16 dimensions) makes a position's self-similarity 1.
print("Similarity of position 10 to positions 9, 10, 11, 30:",
      [round(float(PE[10] @ PE[p] / 8), 2) for p in (9, 10, 11, 30)])
# Without this, attention treats the input as an unordered SET of tokens:
perm = rng.permutation(len(tokens))
o1, _ = attention(X, X, X); o2, _ = attention(X[perm], X[perm], X[perm])
print("Shuffling tokens just shuffles outputs (order ignored without positions):",
      np.allclose(o1[perm], o2))

Attention weights for the token 'it':
  cat      0.169
  it       0.146
  mouse    0.125
  hungry   0.116
'it' attends most to the animal words, 'cat' in particular (a predator that is hungry).

head 0: token 'it' attends most strongly to 'hungry'
head 1: token 'it' attends most strongly to 'was'
multi-head output shape: (9, 8) (one vector per token)

Causal attention weights (rows = token, columns = attended token):
[[1.   0.   0.   0.   0.  ]
 [0.27 0.73 0.   0.   0.  ]
 [0.24 0.31 0.45 0.   0.  ]
 [0.31 0.19 0.19 0.31 0.  ]
 [0.16 0.26 0.16 0.16 0.26]]
Everything above the diagonal is zero: no token sees the future.

d_k=    4: largest weight without scaling 0.667, with scaling 0.340
d_k=   64: largest weight without scaling 0.785, with scaling 0.282
d_k= 1024: largest weight without scaling 0.995, with scaling 0.281
Without scaling, large dimensions make softmax nearly one-hot and gradients vanish.

Positional encodings give each position a unique pattern.
Similarity of position 10

**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Attention weights for the token 'it':
  cat      0.169
  it       0.146
  mouse    0.125
  hungry   0.116
'it' attends most to the animal words, 'cat' in particular (a predator that is hungry).
 
head 0: token 'it' attends most strongly to 'hungry'
head 1: token 'it' attends most strongly to 'was'
multi-head output shape: (9, 8) (one vector per token)
 
Causal attention weights (rows = token, columns = attended token):
[[1.   0.   0.   0.   0.  ]
 [0.27 0.73 0.   0.   0.  ]
 [0.24 0.31 0.45 0.   0.  ]
 [0.31 0.19 0.19 0.31 0.  ]
 [0.16 0.26 0.16 0.16 0.26]]
Everything above the diagonal is zero: no token sees the future.
 
d_k=    4: largest weight without scaling 0.667, with scaling 0.340
d_k=   64: largest weight without scaling 0.785, with scaling 0.282
d_k= 1024: largest weight without scaling 0.995, with scaling 0.281
Without scaling, large dimensions make softmax nearly one-hot and gradients vanish.
 
Positional encodings give each position a unique pattern.
Similarity of position 10 to positions 9, 10, 11, 30: [0.94, 1.0, 0.94, 0.72]
Shuffling tokens just shuffles outputs (order ignored without positions): True
```

**Investigate**

1. Change the embedding of "hungry" to [0, 0, 0, 1] (a function word). How does the attention of "it" change?
2. Prove the permutation result mathematically: why does shuffling the inputs to attention simply shuffle its outputs?
3. Add positional encodings to X before computing attention and rerun the shuffling test. What changes?

## 7. Compression, denoising and anomaly detection

**File:** `l4_autoencoder.py`

In [8]:
"""
Layer 4 example: an AUTOENCODER learns to compress and reconstruct data.
 
File: l4_autoencoder.py
 
Purpose:
    An autoencoder squeezes each input through a narrow BOTTLENECK and then
    tries to rebuild it. To succeed it must learn the most important
    structure in the data. It is trained with the INPUT as its own TARGET,
    so no labels are needed (unsupervised / self-supervised). Uses include
    compression, denoising and anomaly detection: inputs unlike the training
    data reconstruct badly.
 
How it works:
    Architecture: 64 pixels -> 32 -> 8 (the 'bottleneck' code) -> 32 -> 64
    1. Train the autoencoder on 1,400 digit images and compare its error
       with PCA, the classic linear compression method, using 8 numbers.
    2. Train a DENOISING autoencoder: noisy images in, clean images out.
    3. Use reconstruction error to flag random images as anomalies.
 
What to look for in the output:
    - Reconstruction error 0.0321 for the autoencoder vs 0.0245 for PCA: on
      this small dataset the simple linear method does slightly better.
    - Denoising autoencoder: error 0.0188, about half of the noisy input's
      0.0370, while the plain autoencoder (0.0385) does not remove noise.
    - Random images reconstruct with error 0.1942 vs 0.0321 for digits, so
      100% are flagged as anomalies and only 1.5% of real digits are.
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
import warnings
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
# Hide convergence warnings to keep the output clean.
warnings.filterwarnings("ignore")
 
rng = np.random.default_rng(0)
# 8x8 digit images as rows of 64 pixels, scaled to 0..1.
X, y = load_digits(return_X_y=True)
X = X / 16.0
# Random split: 1,400 training images, the rest for testing.
idx = rng.permutation(len(X)); tr, te = idx[:1400], idx[1400:]
 
# ---- Train the autoencoder -----------------------------------------------------------
# An MLP regressor with layers 32 -> 8 -> 32. The 8-unit middle layer is the
# bottleneck: every image must be described by just 8 numbers (the "code").
# The first half is the ENCODER, the second half the DECODER.
ae = MLPRegressor(hidden_layer_sizes=(32, 8, 32), activation="relu", max_iter=2000,
                  learning_rate_init=0.003, random_state=0)
ae.fit(X[tr], X[tr])                             # input == target
 
# Mean squared error per image: average of (pixel difference)^2 across 64 pixels.
mse = lambda a, b: ((a - b) ** 2).mean(axis=1)
recon = ae.predict(X[te])
print(f"Autoencoder (8-number code) reconstruction error: "
      f"{mse(recon, X[te]).mean():.4f}")
# PCA with 8 components is a LINEAR autoencoder; a useful baseline.
pca = PCA(8).fit(X[tr])
print(f"PCA (8 components) reconstruction error:          "
      f"{mse(pca.inverse_transform(pca.transform(X[te])), X[te]).mean():.4f}")
 
# Show one digit before and after (as ASCII art)
# Each pixel value 0..1 picks one of 9 characters, from ' ' (dark) to '@'.
def ascii(v):
    return ["".join(" .:-=+*#@"[min(8, int(p * 9))] for p in v.reshape(8, 8)[r])
            for r in range(8)]
print(f"\nOriginal digit {y[te[0]]}   Reconstruction")
for a, b in zip(ascii(X[te[0]]), ascii(np.clip(recon[0], 0, 1))):
    print(f"  {a}        {b}")
 
# ---- Denoising autoencoder: train on NOISY inputs with CLEAN targets ---------------
# Because the target is the clean image, the network must learn what digits
# really look like and treat the noise as something to remove.
noisy_tr = np.clip(X[tr] + rng.normal(0, 0.25, X[tr].shape), 0, 1)
dae = MLPRegressor(hidden_layer_sizes=(128, 32, 128), max_iter=2000,
                   learning_rate_init=0.003, random_state=0).fit(noisy_tr, X[tr])
# Fresh noise on the test images, then compare all three against the clean truth.
noisy_te = np.clip(X[te] + rng.normal(0, 0.25, X[te].shape), 0, 1)
print(f"\nNoisy test images, error vs clean:          "
      f"{mse(noisy_te, X[te]).mean():.4f}")
print(f"Plain autoencoder output, error vs clean:   "
      f"{mse(np.clip(ae.predict(noisy_te), 0, 1), X[te]).mean():.4f}")
print(f"Denoising autoencoder output, error vs clean: "
      f"{mse(np.clip(dae.predict(noisy_te), 0, 1), X[te]).mean():.4f}")
 
# ---- Anomaly detection: random noise images are not digits ------------------------
# The autoencoder only learned to rebuild digits, so anything else comes back
# badly. A high reconstruction error therefore signals "unusual input".
fake = rng.random((100, 64))
print(f"\nMean reconstruction error, real digits:   "
      f"{mse(ae.predict(X[te]), X[te]).mean():.4f}")
print(f"Mean reconstruction error, random images: "
      f"{mse(ae.predict(fake), fake).mean():.4f}")
# Threshold: the error that 99% of training digits stay below.
threshold = np.percentile(mse(ae.predict(X[tr]), X[tr]), 99)
print(f"Flag as anomaly if error > {threshold:.4f}: "
      f"{(mse(ae.predict(fake), fake) > threshold).mean()*100:.0f}% "
      f"of random images flagged, "
      f"{(mse(ae.predict(X[te]), X[te]) > threshold).mean()*100:.1f}% "
      f"of real digits flagged")

Autoencoder (8-number code) reconstruction error: 0.0321
PCA (8 components) reconstruction error:          0.0245

Original digit 2   Reconstruction
   .#@+             @@=   
   +@#@.           =@##   
   @::@            ===@   
     :@.            .=*   
     ##              *+   
    +@:             .#+   
   :@#=++          .##@*- 
   .@@#*+           @@@*+.



Noisy test images, error vs clean:          0.0370
Plain autoencoder output, error vs clean:   0.0385
Denoising autoencoder output, error vs clean: 0.0188

Mean reconstruction error, real digits:   0.0321
Mean reconstruction error, random images: 0.1942
Flag as anomaly if error > 0.0697: 100% of random images flagged, 1.5% of real digits flagged


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Autoencoder (8-number code) reconstruction error: 0.0321
PCA (8 components) reconstruction error:          0.0245
 
Original digit 2   Reconstruction
   .#@+             @@=   
   +@#@.           =@##   
   @::@            ===@   
     :@.            .=*   
     ##              *+   
    +@:             .#+   
   :@#=++          .##@*- 
   .@@#*+           @@@*+.
 
Noisy test images, error vs clean:          0.0370
Plain autoencoder output, error vs clean:   0.0385
Denoising autoencoder output, error vs clean: 0.0188
 
Mean reconstruction error, real digits:   0.0321
Mean reconstruction error, random images: 0.1942
Flag as anomaly if error > 0.0697: 100% of random images flagged, 1.5% of real digits flagged
```

**What the output shows**

Three findings are worth noting. On this small, simple dataset PCA reconstructs slightly better than the autoencoder, a reminder that deep models do not automatically beat classical methods. A plain autoencoder does not remove noise, but the same architecture trained on noisy inputs with clean targets halves the error. And reconstruction error separates real digits from random images perfectly, the basis of autoencoder anomaly detection.

**Investigate**

1. Change the bottleneck from 8 to 2 and plot the 2-dimensional codes of the test digits coloured by label. Compare with the PCA plot in Chapter 3.
2. Train the anomaly detector only on digits 0 to 8, then measure reconstruction error on digit 9. Is 9 detected as an anomaly?

## 8. Transfer learning and augmentation

**File:** `l4_transfer_augment.py`

In [9]:
"""
Layer 4 example: TRANSFER LEARNING and DATA AUGMENTATION on handwritten digits.
 
File: l4_transfer_augment.py
 
Purpose:
    Deep networks need lots of labelled data. Two standard tricks reduce that
    need. Transfer learning reuses features a network already learned on one
    task for a new task. Data augmentation creates extra training examples
    by transforming the ones we have, teaching the model what should NOT
    change its answer.
 
How it works:
    Part 1 (transfer): train a network to recognise digits 0-9, then reuse its
    hidden layer as a FEATURE EXTRACTOR for a different task (is the digit
    prime?) where we have only a handful of labelled examples. The pretrained
    layer is FROZEN and only a small new classifier (the 'head') is trained.
 
    Part 2 (augmentation): create shifted copies of training images and show
    the model becomes robust to shifts it would otherwise fail on.
 
What to look for in the output:
    - With only 10 labels, transferred features score 0.833 vs 0.750 for raw
      pixels; by 100 labels the gap shrinks (0.915 vs 0.905).
    - Augmentation grows the training set from 1200 to 10800 examples.
    - On images shifted 1 right and 1 down, the plain model drops to 0.137
      while the augmented model keeps 0.953.
    - On original images the augmented model is slightly lower (0.955 vs
      0.975): a small price for robustness.
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
import warnings
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
# Hide convergence warnings to keep the output clean.
warnings.filterwarnings("ignore")
 
rng = np.random.default_rng(0)
# 1,797 digit images as rows of 64 pixels, scaled to 0..1.
X, y = load_digits(return_X_y=True)
X = X / 16.0
 
# ================= PART 1: TRANSFER LEARNING =================
# Split images into two separate groups so no image is shared between tasks.
idx = rng.permutation(len(X))
A, B = idx[:900], idx[900:]
 
# SOURCE task (plenty of labels): recognise which digit 0-9 an image shows
# Its 64 hidden units learn general stroke and shape detectors.
base = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500,
                     random_state=0).fit(X[A], y[A])
 
# Run images through the pretrained first layer only. Its weights are never
# updated again: this is what "freezing" a layer means.
def features(data):
    """Hidden layer activations = features learned on the source task
    (ReLU of layer 1)."""
    return np.maximum(0, data @ base.coefs_[0] + base.intercepts_[0])
 
# TARGET task (few labels): is the digit a PRIME number (2, 3, 5, 7)?
# This is a different question, answered on different images.
prime = np.isin(y, [2, 3, 5, 7]).astype(int)
Xb, yb = X[B], prime[B]
 
# Compare three approaches for n = 10, 20, 50, 100 labelled images.
print("Target task 'is the digit prime?' with only n labelled images:")
print(f"{'n':>5s} {'raw pixels':>11s} {'MLP from scratch':>17s} "
      f"{'transferred features':>21s}")
for n in (10, 20, 50, 100):
    # Pick n training images; everything else in group B is the test set.
    train_idx = np.random.default_rng(0).choice(len(B), n, replace=False)
    test_mask = np.ones(len(B), bool); test_mask[train_idx] = False
    # Baseline 1: a linear classifier on raw pixels.
    raw = LogisticRegression(max_iter=3000).fit(Xb[train_idx], yb[train_idx])
    # Baseline 2: a whole new network trained only on the n images.
    scratch = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=0)
    scratch.fit(Xb[train_idx], yb[train_idx])
    # FREEZE the pretrained layer; train only a small new classifier (the 'head') on top
    # Few parameters to fit, so a handful of labels is enough.
    head = LogisticRegression(max_iter=3000).fit(features(Xb[train_idx]),
                                                 yb[train_idx])
    print(f"{n:5d} {raw.score(Xb[test_mask], yb[test_mask]):11.3f} "
          f"{scratch.score(Xb[test_mask], yb[test_mask]):17.3f} "
          f"{head.score(features(Xb[test_mask]), yb[test_mask]):21.3f}")
print("The benefit is largest when labelled data is scarce, and shrinks as n grows.\n")
 
# ================= PART 2: DATA AUGMENTATION =================
# New random split: 1,200 training images, the rest for testing.
perm = rng.permutation(len(X))
tr, te = perm[:1200], perm[1200:]
imgs = X.reshape(-1, 8, 8)                      # (N, height, width)
 
# Shift a batch of (N, 8, 8) images: copy the part that stays inside the
# frame to its new position; the uncovered strip stays zero (background).
def shift(batch, dx, dy):
    """Move images by dx, dy pixels, filling the gap with zeros
    (not wrapping round)."""
    out = np.zeros_like(batch)
    src = batch[:, max(0, -dy):8 - max(0, dy), max(0, -dx):8 - max(0, dx)]
    out[:, max(0, dy):max(0, dy) + src.shape[1],
        max(0, dx):max(0, dx) + src.shape[2]] = src
    return out
 
# Augmented training set: original plus 8 shifted copies
# A shifted 3 is still a 3, so each copy keeps its original label.
offsets = [(dx, dy) for dx in (-1, 0, 1) for dy in (-1, 0, 1)]
X_aug = np.concatenate([shift(imgs[tr], dx, dy).reshape(len(tr), -1)
                        for dx, dy in offsets])
y_aug = np.tile(y[tr], len(offsets))
 
# Same network trained twice: once on originals, once on the augmented set.
plain = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500,
                      random_state=0).fit(X[tr], y[tr])
aug = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500,
                    random_state=0).fit(X_aug, y_aug)
 
# Test both on original and shifted test images. A dense network treats each
# pixel position separately, so a 1-pixel shift looks like a new image to it.
print(f"Training examples: plain {len(tr)}, augmented {len(X_aug)}")
print(f"{'test images':28s} {'plain':>6s} {'augmented':>10s}")
for label, (dx, dy) in [("original", (0, 0)), ("shifted 1 pixel right", (1, 0)),
                        ("shifted 1 pixel up", (0, -1)),
                        ("shifted 1 right and 1 down", (1, 1))]:
    Xt = shift(imgs[te], dx, dy).reshape(len(te), -1)
    print(f"{label:28s} {plain.score(Xt, y[te]):6.3f} {aug.score(Xt, y[te]):10.3f}")

Target task 'is the digit prime?' with only n labelled images:
    n  raw pixels  MLP from scratch  transferred features
   10       0.750             0.778                 0.833


   20       0.727             0.772                 0.807


   50       0.850             0.893                 0.903


  100       0.905             0.911                 0.915
The benefit is largest when labelled data is scarce, and shrinks as n grows.



Training examples: plain 1200, augmented 10800
test images                   plain  augmented
original                      0.975      0.955
shifted 1 pixel right         0.486      0.960
shifted 1 pixel up            0.456      0.970
shifted 1 right and 1 down    0.137      0.953


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Target task 'is the digit prime?' with only n labelled images:
    n  raw pixels  MLP from scratch  transferred features
   10       0.750             0.778                 0.833
   20       0.727             0.772                 0.807
   50       0.850             0.893                 0.903
  100       0.905             0.911                 0.915
The benefit is largest when labelled data is scarce, and shrinks as n grows.
 
Training examples: plain 1200, augmented 10800
test images                   plain  augmented
original                      0.975      0.955
shifted 1 pixel right         0.486      0.960
shifted 1 pixel up            0.456      0.970
shifted 1 right and 1 down    0.137      0.953
```

**What the output shows**

With only 10 labelled examples, features transferred from the digit recogniser give 83 percent accuracy against 75 percent for raw pixels, and the advantage shrinks as labelled data grows, exactly the pattern seen in practice. The augmentation results are even more striking: a network that scores 98 percent on centred digits drops to 14 percent when images move one pixel diagonally, while the network trained with shifted copies stays above 95 percent.

**Investigate**

1. Try a target task of "is the digit greater than 4?". Does transfer still help?
2. Transfer can also hurt, known as negative transfer. Pretrain the base network on only digits 0 and 1 and repeat the prime experiment. What happens?
3. Add small rotations to the augmentation using scipy.ndimage.rotate and test on rotated digits.

## 9. CNNs and pretrained models in Keras

**File:** `l4_keras_cnn.py`

The following listing trains a CNN on the full 28 by 28 MNIST dataset with augmentation, batch normalisation, dropout and early stopping, then builds a transfer learning model on a pretrained ResNet50. As in Chapter 4, TensorFlow could not be installed in the environment used to build this guide, so this listing has not been run; it follows the documented Keras 3 API and should be tested before use.

> **Not run in this notebook.** This program needs TensorFlow/Keras and a download of MNIST and the ResNet50 weights. Install what it needs and run it on your own machine; the code is included so that you can read and adapt it.

In [ ]:
"""
Layer 4 framework example: a convolutional network on the full MNIST dataset
(60,000 training images, 28x28 pixels) in Keras, followed by transfer learning
with a pretrained ResNet50.
 
File: l4_keras_cnn.py
 
Purpose:
    The NumPy examples show how a CNN works inside; this shows how it is
    built in practice. A framework handles the gradients, GPU kernels and
    training loop, so a modern CNN with augmentation, batch normalisation and
    dropout fits in a few lines. Part 2 reuses a network pretrained on
    ImageNet, the usual starting point for real image tasks.
 
How it works:
    1. Load MNIST, add a channel dimension and scale pixels to 0..1.
    2. Build: augmentation -> two conv/batch-norm/pool blocks -> dense layers.
    3. Train with AdamW and early stopping, then report test accuracy.
    4. Load ResNet50 without its top layer, freeze it, and add a new
       5-class head for your own photos.
 
What to look for in the output:
    - model.summary() shows the shapes shrinking 28x28 -> 14x14 -> 7x7 while
      the number of channels grows 32 -> 64.
    - Validation accuracy per epoch; training may stop early if it stops
      improving for 2 epochs.
    - A test accuracy of around 99% is typical (not verified here).
    - In transfer.summary(), almost all parameters are non-trainable
      (frozen); only the new Dense(5) head is trained.
 
Requires:
    pip install tensorflow
    Note: this listing was not executed in the environment used to build this
    guide. Typical results reported for this architecture are around 99% test
    accuracy; run it and record your own figures.
"""
 
import numpy as np
import keras
from keras import layers
 
# ============ Part 1: a CNN trained from scratch on MNIST ============
# Images arrive as (N, 28, 28) integers 0..255.
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
# add channel dimension, scale 0-1: (N, 28, 28) -> (N, 28, 28, 1)
# (Keras uses (batch, height, width, channels) order.)
x_train = x_train[..., None].astype("float32") / 255.0
x_test = x_test[..., None].astype("float32") / 255.0
 
# Layers run top to bottom. Conv layers use 3x3 filters shared across the
# image; padding="same" keeps the size, pooling halves it.
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    # Randomly shifts images by up to 10% each batch, so the model sees a new
    # variant every epoch.
    # data augmentation, active only during training
    layers.RandomTranslation(0.1, 0.1),
    # 32 learned filters -> output (28, 28, 32).
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    # Normalises each channel over the batch, keeping activations well scaled.
    layers.BatchNormalization(),
    layers.MaxPooling2D(),                     # 28x28 -> 14x14
    # 64 filters, each looking across all 32 channels below -> (14, 14, 64).
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),                     # 14x14 -> 7x7
    # (7, 7, 64) -> a vector of 3,136 features.
    layers.Flatten(),
    # Dropout zeroes 30% of features at random in training to reduce overfitting.
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    # Softmax gives a probability for each of the 10 digits.
    layers.Dense(10, activation="softmax"),
])
# prints layer shapes and parameter counts
model.summary()
 
# AdamW: the Adam optimiser with weight decay. "sparse" cross-entropy accepts
# integer labels 0..9 rather than one-hot vectors.
model.compile(optimizer=keras.optimizers.AdamW(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# Stop if validation loss fails to improve for 2 epochs; keep the best weights.
callbacks = [keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
# Hold back 10% of the training data for validation; 128 images per batch.
model.fit(x_train, y_train, validation_split=0.1, epochs=10, batch_size=128,
          callbacks=callbacks)
print("MNIST test accuracy:", model.evaluate(x_test, y_test, verbose=0)[1])
 
# ============ Part 2: transfer learning with a pretrained ResNet50 ============
# Downloads weights learned on ImageNet (about 100 MB) the first time.
# include_top=False drops the 1,000-class ImageNet classifier; pooling="avg"
# turns the final feature maps into one 2,048-number vector per image.
base = keras.applications.ResNet50(weights="imagenet", include_top=False,
                                   input_shape=(224, 224, 3), pooling="avg")
# Frozen weights are not updated, so the general features learned from
# millions of images are kept and only the new head needs data.
# FREEZE the pretrained feature extractor
base.trainable = False
 
inputs = keras.Input(shape=(224, 224, 3))
# Scale pixels exactly as ResNet50 saw them during its original training.
x = keras.applications.resnet50.preprocess_input(inputs)
# training=False also keeps batch normalisation in inference mode.
x = base(x, training=False)
outputs = layers.Dense(5, activation="softmax")(x)   # new head for YOUR 5 classes
transfer = keras.Model(inputs, outputs)
transfer.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])
transfer.summary()
# Load your own images with keras.utils.image_dataset_from_directory("photos/",
# image_size=(224, 224)) where each sub-folder is one class, then call
# transfer.fit(...).

## 10. Fooling a network

**File:** `l4_adversarial.py`

In [10]:
"""
Layer 4 example: ADVERSARIAL EXAMPLES with the fast gradient sign method
(Goodfellow et al., 2015).
 
File: l4_adversarial.py
 
Purpose:
    Backpropagation can compute the gradient of the loss with respect to the
    INPUT PIXELS instead of the weights. Nudging every pixel a tiny amount in
    the direction that INCREASES the loss can flip the prediction, even though
    the image barely changes to a human. This reveals that networks can rely
    on patterns very different from those people use, which matters for the
    safety and security of deployed systems.
 
How it works:
    1. Train a small MLP on digit images.
    2. Backpropagate to get d loss / d pixel for every test image.
    3. Fast gradient sign method (FGSM): add epsilon * sign(gradient) to each
       pixel, for several values of epsilon, and measure accuracy.
    4. Compare with random +/- noise of the same size.
 
What to look for in the output:
    - Clean accuracy is 0.919; at epsilon 0.2 it collapses to 0.008.
    - Random noise of the same size only lowers accuracy to 0.856.
    - The example digit 2 is predicted as 9 after the attack.
 
Requires:
    pip install numpy scikit-learn
"""
 
import numpy as np
import warnings
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
# Hide convergence warnings to keep the output clean.
warnings.filterwarnings("ignore")
 
# ---- Train the model we will attack --------------------------------------------------
# Pixels scaled to 0..1; first 1,400 images train, the rest test.
X, y = load_digits(return_X_y=True)
X = X / 16.0
model = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500,
                      random_state=0).fit(X[:1400], y[:1400])
Xt, yt = X[1400:], y[1400:]
print(f"Clean test accuracy: {model.score(Xt, yt):.3f}")
 
# Pull out the trained weights: W1 (64, 64) input -> hidden, W2 (64, 10)
# hidden -> output, plus their biases.
W1, W2 = model.coefs_; b1, b2 = model.intercepts_
 
 
# ---- input_gradient(): backpropagation all the way to the pixels ---------------------
# The usual backward pass, but instead of stopping at the weights we carry on
# one more step to get the gradient for each input pixel. The weights stay fixed.
def input_gradient(x, label):
    """Gradient of cross-entropy loss with respect to the input pixels."""
    # Forward pass: hidden layer with ReLU, then softmax probabilities.
    z1 = x @ W1 + b1; a1 = np.maximum(0, z1)
    s = a1 @ W2 + b2
    p = np.exp(s - s.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    # Backward pass through the output layer, the ReLU and the first layer.
    ds = p.copy(); ds[np.arange(len(x)), label] -= 1      # d loss / d scores
    da1 = ds @ W2.T
    dz1 = da1 * (z1 > 0)
    return dz1 @ W1.T                                     # d loss / d input
 
 
# (N_test, 64): for each pixel, which way makes the model more wrong.
grad = input_gradient(Xt, yt)
print(f"\n{'epsilon':>8s} {'max pixel change':>17s} {'accuracy':>9s}")
# FGSM uses only the SIGN of the gradient, so every pixel moves by exactly
# epsilon in its most harmful direction. Small changes to all 64 pixels add
# up to a large change in the model's scores. Clip keeps pixels in 0..1.
for eps in (0.0, 0.05, 0.1, 0.15, 0.2):
    # FGSM: step in sign of gradient
    adv = np.clip(Xt + eps * np.sign(grad), 0, 1)
    print(f"{eps:8.2f} {np.abs(adv - Xt).max():17.2f} {model.score(adv, yt):9.3f}")
 
# Compare with RANDOM noise of the same size
# Random signs point in random directions, so their effects on the score
# largely cancel out; the gradient signs all push the same way.
rng = np.random.default_rng(0)
rand = np.clip(Xt + 0.2 * rng.choice([-1, 1], Xt.shape), 0, 1)
print(f"\nRandom +/-0.2 noise (same size as eps=0.2): "
      f"accuracy {model.score(rand, yt):.3f}")
print("Random noise barely matters; carefully chosen noise is devastating.")
 
# ---- One concrete example ------------------------------------------------------------
# Attack the first test image (kept 2D with i:i+1, as the model expects).
i = 0
adv = np.clip(Xt[i:i+1] + 0.2 * np.sign(grad[i:i+1]), 0, 1)
print(f"\nExample: true digit {yt[i]}, clean prediction "
      f"{model.predict(Xt[i:i+1])[0]}, "
      f"adversarial prediction {model.predict(adv)[0]}")

Clean test accuracy: 0.919

 epsilon  max pixel change  accuracy
    0.00              0.00     0.919
    0.05              0.05     0.743
    0.10              0.10     0.446
    0.15              0.15     0.126
    0.20              0.20     0.008

Random +/-0.2 noise (same size as eps=0.2): accuracy 0.856
Random noise barely matters; carefully chosen noise is devastating.

Example: true digit 2, clean prediction 2, adversarial prediction 9


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Clean test accuracy: 0.919
 
 epsilon  max pixel change  accuracy
    0.00              0.00     0.919
    0.05              0.05     0.743
    0.10              0.10     0.446
    0.15              0.15     0.126
    0.20              0.20     0.008
 
Random +/-0.2 noise (same size as eps=0.2): accuracy 0.856
Random noise barely matters; carefully chosen noise is devastating.
 
Example: true digit 2, clean prediction 2, adversarial prediction 9
```

**What the output shows**

Changing each pixel by at most 0.2 in the adversarial direction reduces accuracy from 92 percent to under 1 percent, while random changes of exactly the same size leave accuracy at 86 percent. The network's decision depends on features that differ from those a person relies on.

**Investigate**

1. Adversarial training: add FGSM examples (with correct labels) to the training set, retrain, and measure accuracy under attack again.
2. Perform a targeted attack that makes every image be classified as the digit 0 by stepping to decrease the loss for label 0.